# Evaluation and Metrics

Training loss tells you how well the model fits training data.
Evaluation metrics tell you how well it generalizes to new data.

For LLM post-training specifically, evaluation is critical:
- **Perplexity** — how well does the model predict text?
- **Accuracy** — for classification / reward models
- **BLEU/ROUGE** — for generation quality
- **Win rate** — for preference alignment (DPO/RLHF)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

# --- Generic eval loop ---

def evaluate(model, loader, loss_fn, device):
    model.eval()   # CRITICAL — disables dropout, fixes batchnorm
    total_loss = 0
    correct    = 0
    total      = 0

    with torch.no_grad():   # CRITICAL — no gradient tracking = faster + less memory
        for X, y in loader:
            X, y    = X.to(device), y.to(device)
            logits  = model(X)
            loss    = loss_fn(logits, y)

            total_loss += loss.item() * len(y)
            correct    += (logits.argmax(dim=1) == y).sum().item()
            total      += len(y)

    return total_loss / total, correct / total


# Quick sanity check
device  = 'cuda' if torch.cuda.is_available() else 'cpu'
model   = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
model   = model.to(device)
loss_fn = nn.CrossEntropyLoss()

X = torch.randn(100, 10)
y = torch.randint(0, 3, (100,))
loader = DataLoader(TensorDataset(X, y), batch_size=16)

val_loss, val_acc = evaluate(model, loader, loss_fn, device)
print(f'Val loss: {val_loss:.4f} | Val acc: {val_acc:.3f}')

## Perplexity — the key metric for language models

**Math:** `PPL = exp(average cross-entropy loss)`

Perplexity = how many equally likely tokens the model is choosing from at each step.
- PPL = 1 → perfect model (100% confident, always right)
- PPL = vocab_size → random model (uniform distribution over all tokens)
- **GPT-2 on WikiText-103:** ~26. **LLaMA-2-7B:** ~5-7

**For post-training:** lower perplexity on your target domain = better fine-tuned model.
After SFT, you should see PPL drop on your instruction-response format.

In [ ]:
def compute_perplexity(model, tokenizer, texts, device, max_len=512):
    """Compute perplexity on a list of texts."""
    model.eval()
    total_loss  = 0
    total_tokens = 0

    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(
                text,
                return_tensors='pt',
                max_length=max_len,
                truncation=True
            ).to(device)

            # For causal LM: input = tokens[:-1], label = tokens[1:]
            # CrossEntropyLoss over every token position
            outputs = model(**inputs, labels=inputs['input_ids'])

            # outputs.loss = average NLL per token
            n_tokens     = inputs['input_ids'].shape[1]
            total_loss  += outputs.loss.item() * n_tokens
            total_tokens += n_tokens

    avg_nll     = total_loss / total_tokens
    perplexity  = torch.exp(torch.tensor(avg_nll)).item()
    return perplexity


# Usage with a real model:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# model     = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
# tokenizer = AutoTokenizer.from_pretrained('gpt2')
# texts = ['The quick brown fox jumps over the lazy dog.']
# ppl = compute_perplexity(model, tokenizer, texts, device)
# print(f'Perplexity: {ppl:.2f}')

print('Perplexity function defined — run with a real model on Colab')

## Precision, Recall, F1 — for classification and reward models

Used heavily in reward model evaluation (binary: good/bad response).

```
Precision = TP / (TP + FP)  — of predicted positives, how many are actually positive?
Recall    = TP / (TP + FN)  — of actual positives, how many did we catch?
F1        = 2 * P * R / (P + R)  — harmonic mean of precision and recall
```

In [ ]:
def compute_metrics(y_true, y_pred):
    tp = ((y_pred == 1) & (y_true == 1)).sum().float()
    fp = ((y_pred == 1) & (y_true == 0)).sum().float()
    fn = ((y_pred == 0) & (y_true == 1)).sum().float()
    tn = ((y_pred == 0) & (y_true == 0)).sum().float()

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    accuracy  = (tp + tn) / len(y_true)

    return {'accuracy': accuracy.item(), 'precision': precision.item(),
            'recall': recall.item(), 'f1': f1.item()}


y_true = torch.tensor([1, 0, 1, 1, 0, 1, 0, 0, 1, 1])
y_pred = torch.tensor([1, 0, 1, 0, 0, 1, 1, 0, 1, 0])

metrics = compute_metrics(y_true, y_pred)
for k, v in metrics.items():
    print(f'{k}: {v:.3f}')

## Overfitting detection — train vs val loss gap

This is THE most important evaluation signal during training.

```
Healthy:     train loss ↓, val loss ↓ — model is generalizing
Overfitting: train loss ↓, val loss ↑ — memorizing training data
Underfitting: both losses high — model too simple or lr too low
```

For LLM fine-tuning: overfitting happens fast (especially with small datasets).
Signs: val loss goes up, model starts repeating training phrases verbatim.

In [ ]:
# Early stopping — stop training when val loss stops improving

class EarlyStopping:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float('inf')
        self.counter    = 0
        self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True


# Simulate val losses over epochs
val_losses = [1.2, 1.0, 0.85, 0.82, 0.81, 0.83, 0.84, 0.86]
early_stop = EarlyStopping(patience=3)

for epoch, vl in enumerate(val_losses):
    early_stop.step(vl)
    status = 'STOP' if early_stop.should_stop else 'continue'
    print(f'Epoch {epoch+1} | val_loss: {vl:.2f} | {status}')
    if early_stop.should_stop:
        break